# 1.49B 1-Layer DeepSeek-V4.1-Flash LoopMoE — Colab TPU v6e-1 Training (`main`)

- **Explicit Config**: [`configs/v6e_1_train.json`](configs/v6e_1_train.json)（無隱藏預設參數，`--config` 主動載入）
- **Backbone**: `d_model=2048`, `V=32,768` (tied), `2-stream closed-form mHC`, `2048×1024` 1024-aligned Engram (`541.1M`, `RowAdamW`), `MLA + YaRN` (Fused Splash Attention, 8-bank Q/O + Loop-0 KV reuse), `128×1024 top-4 + 1 shared` MoE (`Pallas` DMA + `Tokamax`), `bptt_window=8`。
- **Curriculum (`v6e-1` 32 GB HBM)**: `R=1 (4K B=16 → 8K B=8 → 16K B=2 → 32K B=1, remat=False)` → `R=2,3,...,32 (32K B=1, remat_moe=True, remat_sublayers=False)`。
- **Lookahead Time-Efficiency Switching**: `R=1, 4K` 先跑 `warmup=1000` 步（warmup 期間不 lookahead）；之後每輪 `main_steps` + `probe_steps`，以**下一階段訓練 `ce_loss` 的每秒下降率**（`第2段頭 - 第1段尾` vs `第2段頭 - 第2段尾`）自動決定是否切換。
- **Checkpoints & XLA Cache**: 本地或掛載 Google Drive 保存。

In [ ]:
# 1. Clone/update repository and install latest pinned stack from requirements.txt
import os
REPO_DIR = '/content/M3'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/GarD0m/M3.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

!pip install -q -U -r {REPO_DIR}/requirements.txt

In [ ]:
# 2. Verify TPU v6e-1 runtime in a subprocess so the Jupyter kernel does NOT lock libtpu.so
!python -c "import jax; print('Backend:', jax.default_backend()); print('Devices:', jax.devices()); [print(d, d.memory_stats()) for d in jax.local_devices()]"

In [ ]:
# 3. Explicit config, tokenizer, checkpoints, and JAX compilation cache paths
CONFIG_PATH = f'{REPO_DIR}/configs/v6e_1_train.json'
TOKENIZER_PATH = f'{REPO_DIR}/artifacts/tokenizer_32k_en/tokenizer.json'
CKPT_DIR = f'{REPO_DIR}/checkpoints'
CACHE_DIR = f'{REPO_DIR}/jax_cache'

# Optional: persist local checkpoints & compilation cache to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# CKPT_DIR = '/content/drive/MyDrive/loopmoe_checkpoints'
# CACHE_DIR = '/content/drive/MyDrive/loopmoe_jax_cache'

In [ ]:
# 4. Quick TPU v6e-1 sanity benchmark (synthetic tokens, Stage 1 only, 5 steps)
%cd {REPO_DIR}
!python scripts/train_tpu.py \
    --config {CONFIG_PATH} \
    --data-mode synthetic \
    --ckpt-dir {CKPT_DIR} \
    --cache-dir {CACHE_DIR} \
    --start-stage 1 --end-stage 1 \
    --max-steps-per-stage 5 \
    --log-every 1

In [ ]:
# 5. Full Config-Driven L3 Curriculum Training with Lookahead (R=1: 4K->32K, then R=2..32: 32K)
%cd {REPO_DIR}
!python scripts/train_tpu.py \
    --config {CONFIG_PATH} \
    --data-mode stream \
    --tokenizer {TOKENIZER_PATH} \
    --ckpt-dir {CKPT_DIR} \
    --cache-dir {CACHE_DIR} \
    --log-every 10